### CC3085 – Inteligencia Artificial
#### Laboratorio 08 – Redes Bayesianas: Localización del Fantasma de Pac-Man

| Campo | Detalle |
|---|---|
| **Curso** | CC3085 – Inteligencia Artificial |
| **Sección** | 21 |
| **Catedrático** | Ing. Alan Gerardo Reyes Figueroa |
| **Fecha** | 2025 |

#### Integrantes
| Nombre | Carné |
|---|---|
| Cindy Gualim | 21226 |
| Javier Linares | 231135 |
| Luis Pedro Lira | 23669 |

---
## Descripción del Problema

Se programa una matriz de probabilidades para localizar al fantasma de Pac-Man en un tablero de **10×10**.

- La red de sensores tiene **5 colores**: rojo, naranja, amarillo, verde y azul.
- La distribución de probabilidad por color se carga desde `Sensor_Color_Distribution.csv`.
- Se asume una **prior uniforme** para todas las celdas del tablero.
- Se actualiza la distribución $\mathbb{P}(G \mid \text{Evidence})$ conforme se recibe evidencia de sensores $\{S_{00}, S_{01}, \ldots, S_{99}\}$.

---
## 1. Importación de Librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import os, csv

---
## 2. Carga de Datos

Se carga la distribución de probabilidad de colores desde `Sensor_Color_Distribution.csv`.  
Cada fila corresponde a una distancia Manhattan `d` (0–18) y cada columna es la probabilidad de observar ese color dado que el fantasma está a distancia `d`.

In [ ]:
CSV_PATH = 'Sensor_Color_Distribution.csv'
COLORS   = ['red', 'orange', 'yellow', 'green', 'blue']
MEANS    = [0, 4, 8, 12, 16]   # distancias centrales de cada color
SIGMA    = 4.0

# Generar el CSV si no existe (distancias 0–18, distribución Gaussiana normalizada)
if not os.path.exists(CSV_PATH):
    rows = [['distance'] + COLORS]
    for d in range(19):
        w = [np.exp(-0.5 * ((d - mu) / SIGMA) ** 2) for mu in MEANS]
        s = sum(w)
        rows.append([d] + [round(x / s, 6) for x in w])
    with open(CSV_PATH, 'w', newline='') as f:
        csv.writer(f).writerows(rows)
    print('CSV generado.')

sensor_df = pd.read_csv(CSV_PATH, index_col='distance')
print(sensor_df.to_string())

**Interpretación:** colores cálidos (rojo, naranja) dominan a distancias cortas; colores fríos (verde, azul) dominan a distancias largas.

---
## 3. Inicialización del Tablero (Prior Uniforme)

Se crea la matriz de probabilidades inicial $\mathbb{P}(G)$ con distribución **uniforme** sobre las 100 celdas del tablero 10×10.

In [ ]:
ROWS, COLS = 10, 10
N_CELLS    = ROWS * COLS

# Prior uniforme P(G = celda) = 1/100 para todas las celdas
prior = np.ones((ROWS, COLS)) / N_CELLS

print(f'Prior uniforme: cada celda tiene P = {prior[0, 0]:.4f}')
print(f'Suma total     = {prior.sum():.6f}')

---
## 4. Función de Actualización Bayesiana

La actualización de $\mathbb{P}(G \mid \text{Evidence})$ usa el Teorema de Bayes:

$$\mathbb{P}(G = g \mid S_{ij} = c) \;\propto\; \mathbb{P}(S_{ij} = c \mid G = g) \cdot \mathbb{P}(G = g)$$

donde la verosimilitud depende de la **distancia Manhattan** entre el sensor $(i,j)$ y la celda candidata $g$:

$$\mathbb{P}(S_{ij} = c \mid G = (r,c')) = P(\text{color}=c \mid d = |i-r| + |j-c'|)$$

In [ ]:
def manhattan(r1, c1, r2, c2):
    return abs(r1 - r2) + abs(c1 - c2)


def bayesian_update(belief, sensor_row, sensor_col, observed_color, sensor_df):
    """
    Actualiza P(G | Evidence) dado que el sensor en (sensor_row, sensor_col)
    reportó observed_color.

    Usa el Teorema de Bayes:
        P(G=g | E) ∝ P(E | G=g) · P(G=g)
    donde la verosimilitud P(E | G=g) = P(color | dist_manhattan(sensor, g)).

    Retorna la nueva creencia normalizada (suma = 1).
    """
    max_d = sensor_df.index.max()
    new_belief = np.zeros_like(belief)

    for r in range(ROWS):
        for c in range(COLS):
            d = min(manhattan(sensor_row, sensor_col, r, c), max_d)
            likelihood = sensor_df.loc[d, observed_color]
            new_belief[r, c] = likelihood * belief[r, c]

    total = new_belief.sum()
    if total > 0:
        new_belief /= total
    return new_belief


def sample_color(ghost_r, ghost_c, sensor_r, sensor_c, sensor_df, rng):
    """Muestrea el color que reporta el sensor dado que el fantasma está en (ghost_r, ghost_c)."""
    d = min(manhattan(ghost_r, ghost_c, sensor_r, sensor_c), sensor_df.index.max())
    probs = sensor_df.loc[d].values.astype(float)
    probs = probs / probs.sum()   # renormalizar para corregir errores de redondeo del CSV
    return rng.choice(sensor_df.columns.tolist(), p=probs)


print('Funciones bayesian_update y sample_color definidas.')

---
## 5. Visualización — Plots Colorizados

### a) Mostrar plots colorizados de la localización

Se muestra cómo cambia $\mathbb{P}(G \mid \text{Evidence})$ conforme se agrega evidencia de sensores.

In [ ]:
# Colormap: azul oscuro (baja prob) → rojo brillante (alta prob)
CMAP = LinearSegmentedColormap.from_list(
    'ghost_cmap',
    ['#0a0a2e', '#1a237e', '#1565c0', '#00897b', '#ffee58', '#ff6f00', '#b71c1c'],
    N=256
)


def plot_belief(belief, title='', ghost_pos=None, sensor_pos=None,
                obs_color=None, ax=None):
    """
    Dibuja la distribución P(G | Evidence) como heatmap sobre el tablero 10x10.

    ghost_pos  : (r, c) posición real del fantasma — marcador triángulo blanco
    sensor_pos : (r, c) último sensor observado    — marcador X blanca
    obs_color  : color observado (para la leyenda)
    """
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(5, 5))

    im = ax.imshow(belief, cmap=CMAP, vmin=0, vmax=max(belief.max(), 1e-9),
                   origin='upper', aspect='equal')

    # Cuadrícula fina
    ax.set_xticks(np.arange(-0.5, COLS, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, ROWS, 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=0.5, alpha=0.3)
    ax.set_xticks(range(COLS))
    ax.set_yticks(range(ROWS))
    ax.tick_params(labelsize=7)

    if ghost_pos is not None:
        ax.plot(ghost_pos[1], ghost_pos[0], 'w^', ms=11, mew=1.5,
                label='Fantasma real', zorder=5)
    if sensor_pos is not None:
        label = f'Sensor ({obs_color})' if obs_color else 'Sensor'
        ax.plot(sensor_pos[1], sensor_pos[0], 'wx', ms=9, mew=2,
                label=label, zorder=5)

    ax.set_title(title, fontsize=8, pad=4)

    if standalone:
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        if ghost_pos or sensor_pos:
            ax.legend(fontsize=7, loc='upper right')
        plt.tight_layout()
        plt.show()
    return im


# Plot del prior uniforme
plot_belief(prior, title='Prior Uniforme — P(G) = 1/100 para todas las celdas')

---
## 6. Celda MAP — Máximo A Posteriori

### b) Indicar la celda más probable donde se encuentra el fantasma

Bajo el argumento de **maximizar la probabilidad a posteriori**:

$$G^* = \arg\max_{g} \; \mathbb{P}(G = g \mid \text{Evidence})$$

In [ ]:
def map_estimate(belief):
    """
    Retorna la celda con mayor probabilidad a posteriori (MAP) y su valor.

    G* = argmax_g P(G=g | Evidence)
    """
    idx = int(np.argmax(belief))
    r, c = divmod(idx, COLS)
    return (r, c), float(belief[r, c])


cell_map, prob_map = map_estimate(prior)
print(f'MAP (prior uniforme): celda {cell_map}  P = {prob_map:.6f}')
print('Con prior uniforme todas las celdas son igualmente probables.')

---
## 7. Simulación Completa

Se coloca al fantasma en una posición secreta y se van recibiendo observaciones de sensores seleccionados aleatoriamente. Tras cada observación se actualiza la creencia y se reporta la celda MAP.

In [ ]:
# ── Escenario 1: sensores aleatorios en el tablero ──────────────────────────────────────────
RNG        = np.random.default_rng(42)
GHOST_TRUE = (3, 7)   # posición real del fantasma (oculta al agente)
N_OBS      = 20
SHOW_STEPS = [0, 2, 5, 9, 14, 19]

sensors = [(RNG.integers(0, ROWS).item(), RNG.integers(0, COLS).item())
           for _ in range(N_OBS)]
colors  = [sample_color(GHOST_TRUE[0], GHOST_TRUE[1], sr, sc, sensor_df, RNG)
           for sr, sc in sensors]

belief       = prior.copy()
history_map  = []
history_prob = []

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
plot_idx = 0

for step in range(N_OBS):
    sr, sc = sensors[step]
    color  = colors[step]
    belief = bayesian_update(belief, sr, sc, color, sensor_df)

    m_cell, m_prob = map_estimate(belief)
    history_map.append(m_cell)
    history_prob.append(m_prob)

    if step in SHOW_STEPS:
        ax = axes[plot_idx]
        im = plot_belief(
            belief,
            title=f'Paso {step+1} | sensor {(sr,sc)} \u2192 {color}\nMAP={m_cell}  P={m_prob:.4f}',
            ghost_pos=GHOST_TRUE, sensor_pos=(sr, sc), obs_color=color, ax=ax
        )
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.legend(fontsize=7, loc='upper right')
        plot_idx += 1

plt.suptitle(f'Escenario 1 \u2014 Evoluci\u00f3n de P(G | Evidence)  [fantasma real: {GHOST_TRUE}]',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

final_map, final_prob = map_estimate(belief)
print(f'Fantasma real  : {GHOST_TRUE}')
print(f'Celda MAP final: {final_map}  con P = {final_prob:.4f}')
print(f'Localizacion correcta: {final_map == GHOST_TRUE}')

In [ ]:
# ── Gráfica de convergencia del MAP ──────────────────────────────────────────────────
fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, N_OBS + 1), history_prob, 'o-', color='#e53935', lw=2, ms=5)
ax1.axhline(1 / N_CELLS, color='gray', ls='--', label='Prior uniforme (0.01)')
ax1.set_xlabel('Numero de observaciones')
ax1.set_ylabel('P(MAP cell | Evidence)')
ax1.set_title('Convergencia de la probabilidad MAP')
ax1.legend()
ax1.grid(True, alpha=0.3)

dists = [manhattan(m[0], m[1], GHOST_TRUE[0], GHOST_TRUE[1]) for m in history_map]
ax2.bar(range(1, N_OBS + 1), dists, color='#1565c0', alpha=0.8)
ax2.set_xlabel('Numero de observaciones')
ax2.set_ylabel('Distancia Manhattan (MAP -> real)')
ax2.set_title('Error de localizacion del estimado MAP')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ── Escenario 2: sensores agrupados cerca del fantasma ──────────────────────────────
print('=== Escenario 2: sensores agrupados cerca del fantasma ===')
GHOST_2 = (6, 2)
RNG2    = np.random.default_rng(7)
belief2 = prior.copy()

close_sensors = [(r, c)
                 for r in range(max(0, GHOST_2[0] - 3), min(ROWS, GHOST_2[0] + 4))
                 for c in range(max(0, GHOST_2[1] - 3), min(COLS, GHOST_2[1] + 4))]

show_at2 = [0, 2, 5, len(close_sensors) - 1]
fig3, axes3 = plt.subplots(1, 4, figsize=(18, 4.5))
plot_idx3 = 0

for step, (sr, sc) in enumerate(close_sensors):
    col2    = sample_color(GHOST_2[0], GHOST_2[1], sr, sc, sensor_df, RNG2)
    belief2 = bayesian_update(belief2, sr, sc, col2, sensor_df)
    m2, p2  = map_estimate(belief2)

    if step in show_at2 and plot_idx3 < 4:
        ax = axes3[plot_idx3]
        im3 = plot_belief(
            belief2,
            title=f'Paso {step+1} | sensor {(sr,sc)} -> {col2}\nMAP={m2}  P={p2:.4f}',
            ghost_pos=GHOST_2, sensor_pos=(sr, sc), obs_color=col2, ax=ax
        )
        plt.colorbar(im3, ax=ax, fraction=0.046, pad=0.04)
        ax.legend(fontsize=7, loc='upper right')
        plot_idx3 += 1

plt.suptitle(f'Escenario 2 \u2014 Sensores cercanos al fantasma  [fantasma real: {GHOST_2}]',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f'Fantasma real  : {GHOST_2}')
print(f'Celda MAP final: {m2}  con P = {p2:.4f}')
print(f'Localizacion correcta: {m2 == GHOST_2}')

---
## 8. Conclusiones

1. **Prior uniforme**: sin ninguna evidencia, todas las celdas del tablero 10×10 tienen la misma probabilidad $P(G) = 0.01$. El heatmap muestra un color homogéneo (azul oscuro uniforme), reflejando incertidumbre total.

2. **Actualización bayesiana**: cada observación de color de un sensor refina la distribución multiplicando la creencia actual por la verosimilitud $P(\text{color} \mid d)$. Colores cálidos (rojo, naranja) favorecen celdas cercanas al sensor; colores fríos (verde, azul) favorecen celdas lejanas.

3. **Convergencia del MAP**: con pocas observaciones el heatmap es difuso y la celda MAP puede no coincidir con el fantasma real. Con ~15–20 sensores la probabilidad se concentra notablemente y el MAP converge a la posición correcta.

4. **Independencia condicional**: el modelo asume que cada sensor $S_{ij}$ es condicionalmente independiente de los demás dado $G$. Esto permite aplicar Bayes de forma secuencial: cada nueva observación multiplica la verosimilitud sin necesidad de guardar el historial completo.

5. **MAP vs. distribución completa**: reportar solo $G^* = \arg\max_g P(G=g \mid E)$ es eficiente, pero puede ser engañoso cuando varias celdas tienen probabilidades similares. El heatmap completo muestra la incertidumbre real del agente y es más informativo en escenarios ambiguos.

6. **Sensores cercanos vs. aleatorios**: el Escenario 2 muestra que sensores agrupados cerca del fantasma concentran la probabilidad más rápidamente que sensores dispersos aleatoriamente, confirmando que la información geográfica de las observaciones es crucial para una localización eficiente.